# PyTorch 注意力接口讲解：nn.MultiheadAttention 与 F.scaled_dot_product_attention

> 本 Notebook 聚焦于讲解 PyTorch 内置的两套注意力计算接口，深入解析参数含义、形状约定与掩码语义。
> - `nn.MultiheadAttention`：**高层模块**，内置 Q/K/V 四个线性投影层（$W_Q, W_K, W_V, W_O$），直接接收原始序列嵌入输入，适合快速搭建标准 Transformer 模型
> - `F.scaled_dot_product_attention`（SDPA）：**底层函数式接口**，只负责缩放点积注意力的纯粹计算，不含任何可学习参数，适合自定义注意力变体
> **说明**：本 Notebook 不做完整训练流程，仅讲解 API 用法与关键概念。读者需对多头注意力机制有基本了解（可参阅 Notebook 2/3）。

## 一、环境准备

In [ ]:
import torch                      # 导入 PyTorch 主库，提供张量计算与自动求导框架
import torch.nn as nn             # 导入神经网络模块，包含 MultiheadAttention 等高层组件
import torch.nn.functional as F   # 导入函数式 API，包含 scaled_dot_product_attention
import math                       # 导入数学库，用于演示手动计算 scale 因子

# 打印 PyTorch 版本，便于环境复现与对照官方文档版本
print('torch version:', torch.__version__)  # 返回值：str，如 '2.12.0+cu132'

# 自动选择计算设备：优先 GPU（cuda:0），无 GPU 时回退 CPU
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')  # 返回值：torch.device
print('device:', device)  # 输出当前使用设备，如 'cuda:0' 或 'cpu'

## 二、接口体系概览

### 2.1 调用链（Calling Chain）

> PyTorch 注意力计算的内部调用层次如下：
> ```
> nn.MultiheadAttention.forward()
>   │
>   ├── 线性投影：W_Q × query、W_K × key、W_V × value
>   │     └── in_proj_weight 划分为 Q/K/V 三块（或独立的 q/k/v_proj_weight）
>   │
>   ├── 分头：将 (N, L, embed_dim) 拆分为 (N, num_heads, L, head_dim)
>   │
>   ├── F.scaled_dot_product_attention()    # 核心注意力计算
>   │     └── 计算 softmax(QK^T / sqrt(d_k)) × V
>   │
>   └── 合并多头 + 输出投影：W_O × concat(heads)
> ```
> **职责划分**：`nn.MultiheadAttention` 负责线性变换、多头管理与输出投影；`F.scaled_dot_product_attention` 只负责注意力权重计算和加权求和，不含任何可学习参数。

### 2.2 使用场景对比

> **形状字母含义**：N = batch size；H = 头数（num_heads）；L = Query 序列长度；S = Key/Value 序列长度；E = 嵌入维度（embed_dim）；E_h = 每头维度（E // H）

> ---

| 场景 | 推荐接口 | 原因 |
|:---|:---|:---|
| 快速搭建标准 Transformer | `nn.MultiheadAttention` | 含投影层，开箱即用 |
| 自定义注意力变体（RoPE、GQA、MQA 等）| `F.scaled_dot_product_attention` | 只负责注意力计算，方便自定义投影 |
| 需要可视化注意力权重 | `nn.MultiheadAttention` (need_weights=True) | 直接返回注意力权重矩阵 |
| 性能敏感场景 | `F.scaled_dot_product_attention` | 支持 Flash Attention / Memory-Efficient 等内核优化 |

> ---

## 三、nn.MultiheadAttention 详解

> `nn.MultiheadAttention` 是 PyTorch 提供的多头注意力完整模块，内置四个可学习投影层。
> - 输入序列内部分别与 $W_Q$、$W_K$、$W_V$ 相乘，得到 Q/K/V 张量
> - 拆分多头后调用 `F.scaled_dot_product_attention` 计算注意力
> - 合并多头结果后再与 $W_O$ 相乘得到输出

### 3.1 构造函数参数

> 以下为 `nn.MultiheadAttention(...)` 构造时的全部可配置参数说明。

> ---

| 参数名 | 类型 | 默认值 | 含义 |
|:---|:---:|:---:|:---|
| `embed_dim` | int | — | 嵌入（模型）维度，Q/K/V 投影的输入输出维度均为此值 |
| `num_heads` | int | — | 多头注意力头数，要求 `embed_dim % num_heads == 0`；每头维度 = `embed_dim // num_heads` |
| `dropout` | float | 0.0 | 注意力权重的 Dropout 概率；仅在训练模式下生效 |
| `bias` | bool | True | 是否为输入投影层和输出投影层添加偏置项 |
| `add_bias_kv` | bool | False | 是否在 K/V 序列末尾追加一个可学习的偏置向量（BERT-style 才需要） |
| `add_zero_attn` | bool | False | 是否在 K/V 序列末尾追加一行全零向量 |
| `kdim` | int/None | None | K 的输入特征维度；None = 与 `embed_dim` 相同（自注意力）；设置不同则启用独立 K 投影层 |
| `vdim` | int/None | None | V 的输入特征维度；None = 与 `embed_dim` 相同；设置不同则启用独立 V 投影层 |
| `batch_first` | bool | False | True = 输入/输出形状为 (N, L, E)；False = (L, N, E)（PyTorch 常规默认） |

> ---

In [ ]:
# ── 构造 nn.MultiheadAttention 实例 ──────────────────────────────────────
MHA = nn.MultiheadAttention(
    embed_dim=256,       # 嵌入（模型）维度，int；Q/K/V 投影的输入输出维度均为 256
    num_heads=8,         # 注意力头数，int；要求 256 % 8 == 0；每头维度 = 256 // 8 = 32
    dropout=0.1,         # 注意力权重的 Dropout 概率，float；训练时随机将部分权重置零
    bias=True,           # bool；为投影层添加偏置项，通常保留
    add_bias_kv=False,   # bool；不在 K/V 末尾追加学习偏置向量（默认 False）
    add_zero_attn=False, # bool；不在 K/V 末尾追加全零向量（默认 False）
    kdim=None,           # K 输入维度，None 表示与 embed_dim 相同（自注意力场景）
    vdim=None,           # V 输入维度，None 表示与 embed_dim 相同
    batch_first=True,    # bool；True 表示输入形状为 (N, L, E)，更符合直觉
)  # 返回值：nn.MultiheadAttention 实例

# ── 打印模型结构 ────────────────────────────────────────────────
print(MHA)
# out_proj 即 W_O 投影层；NonDynamicallyQuantizableLinear 是 Linear 的子类，内部行为相同

# ── 统计可学习参数量 ─────────────────────────────────────────────
print()
for name, p in MHA.named_parameters():  # 遍历所有命名参数，返回 (str, Tensor) 元组
    print(f'  {name}: 形状 {tuple(p.shape)}')  # 打印各参数名称及形状

total_params = sum(p.numel() for p in MHA.parameters())  # 汇总所有层的参数量，返回 int
print(f'\n可学习参数总量：{total_params:,}')  # 3 * embed_dim^2 + embed_dim^2 = 4 * 256^2 = 262,144
# in_proj_weight: (3*256, 256) = 196,608 —— Q/K/V 投影矩阵堆叠
# in_proj_bias:   (3*256,) = 768     —— Q/K/V 偏置堆叠
# out_proj.weight:(256, 256) = 65,536 —— 输出投影矩阵
# out_proj.bias:  (256,) = 256       —— 输出投影偏置

### 3.2 前向传播接口（forward 方法）

> `nn.MultiheadAttention.forward(query, key, value, ...)` 的所有参数说明如下。

> ---

| 参数名 | 形状 | 类型 | 默认值 | 含义 |
|:---|:---:|:---:|:---:|:---|
| `query` | (N, L, E) | Tensor | — | Query 序列；自注意力时与 key/value 相同 |
| `key` | (N, S, E_k) | Tensor | — | Key 序列；E_k = kdim 或 embed_dim |
| `value` | (N, S, E_v) | Tensor | — | Value 序列；E_v = vdim 或 embed_dim |
| `key_padding_mask` | (N, S) | bool/float | None | True=忽略该 Key 位置（padding）；float 直接加到分数 |
| `need_weights` | — | bool | True | True = 返回注意力权重矩阵（有额外开销） |
| `attn_mask` | (L,S) or (N·H,L,S) | bool/float | None | bool: True=忽略；float: 加到注意力分数（-inf=不可见） |
| `average_attn_weights` | — | bool | True | True=返回各头权重均值 (N,L,S)；False=返回每头独立 (N,H,L,S) |
| `is_causal` | — | bool | False | True=自动生成因果掩码；与 attn_mask 互斥 |

> ---

> **返回值**：一个二元组 `(attn_output, attn_output_weights)`

> ---

| 返回值 | 形状 | 条件 |
|:---|:---:|:---|
| `attn_output` | (N, L, embed_dim) | 永远返回 |
| `attn_output_weights` | (N, L, S) | `need_weights=True` 且 `average_attn_weights=True` |
| `attn_output_weights` | (N, H, L, S) | `need_weights=True` 且 `average_attn_weights=False` |
| `attn_output_weights` | None | `need_weights=False` |

> ---

> **`key_padding_mask` 与 `attn_mask` 的使用时机**：
>
> - **TransformerEncoder**：只需传 `key_padding_mask`——双向注意力，所有位置互相可见，无需因果约束
> - **TransformerDecoder 自注意力**：两者都要传——`key_padding_mask` 屏蔽目标序列 padding，`attn_mask` 作为因果掩码防止看到未来 token
> - **TransformerDecoder 交叉注意力**：只需传 `key_padding_mask`（即 `memory_key_padding_mask`）——Key/Value 来自完整的 Encoder 输出，不存在「未来」概念，无需因果约束
>
> 记忆口诀：**哪里有自回归生成，哪里就需要 `attn_mask`（因果掩码）**。

#### 3.2.1 自注意力（Self-Attention）

> 自注意力中，Query、Key、Value 均来自同一个序列。

##### 3.2.1.1 编码器自注意力

> 编码器自注意力中 Q=K=V=x，所有 token 双向可见，无需因果授码；仅需传入 `key_padding_mask` 屏蔽 padding 位置。

In [ ]:
# ── 构造自注意力所需的 MHA 实例 ──────────────────────────────────────────
N_sa, L_sa, E_sa = 2, 12, 256  # N=batch 大小=2, L=序列长度=12, E=嵌入维度=256

mha_sa = nn.MultiheadAttention(  # 构造编码器自注意力 MHA，kdim/vdim 默认 None 即与 embed_dim 相同
    embed_dim=E_sa, num_heads=8, batch_first=True, dropout=0.0,
)  # 返回值：nn.MultiheadAttention 实例
mha_sa.eval()  # 切换评估模式，关闭 Dropout，确保输出确定性

# ── 构造输入序列 ────────────────────────────────────────────────────────────
# 编码器自注意力中 Q=K=V=x，三者均来自同一个源序列
x_sa = torch.rand(N_sa, L_sa, E_sa)  # 形状 (2, 12, 256)，dtype=float32，模拟词嵌入+位置编码后的序列

# ── 构造 padding 掩码 ────────────────────────────────────────────────────────
# key_padding_mask：形状 (N, S) = (2, 12)，dtype=bool；True=该位置是 padding，不参与注意力
# 编码器自注意力只需此掩码，无需因果掩码（所有 token 双向可见）
src_key_padding_mask_sa = torch.zeros(N_sa, L_sa, dtype=torch.bool)  # 初始全 False（无 padding）
src_key_padding_mask_sa[0, 10:] = True  # 第 0 个样本：位置 10、11 是 padding（该句只有 10 个词）

with torch.no_grad():  # 展示演示，不计算梯度
    attn_out_sa, attn_weights_sa = mha_sa(
        query=x_sa,                                # Query 输入，形状 (N, L, E) = (2, 12, 256)
        key=x_sa,                                  # Key 输入，形状 (N, S, E_k) = (2, 12, 256)
        value=x_sa,                                # Value 输入，形状 (N, S, E_v) = (2, 12, 256)
        key_padding_mask=src_key_padding_mask_sa,  # padding 掩码，形状 (N, S) = (2, 12)；True=忽略该 Key
        need_weights=False,                        # bool；False 时 attn_weights 返回 None
    )
    # 返回值 1—attn_out_sa：形状 (N, L, embed_dim) = (2, 12, 256)，dtype=float32
    # 返回值 2—attn_weights_sa：None（need_weights=False）

print('编码器自注意力输入形状:', x_sa.shape)                       # -> torch.Size([2, 12, 256])
print('编码器自注意力输出形状:', attn_out_sa.shape)               # -> torch.Size([2, 12, 256])
print('padding 掩码形状:', src_key_padding_mask_sa.shape)         # -> torch.Size([2, 12])
print('注意力权重:', attn_weights_sa)                              # -> None（need_weights=False）
print('输入形状 = 输出形状？', x_sa.shape == attn_out_sa.shape)   # -> True；自注意力不改变序列长度

##### 3.2.1.2 解码器自注意力

> 解码器自注意力同样 Q=K=V=tgt，但必须同时传入两个掩码：
>
> - `attn_mask`（因果掩码）：防止第 t 个 token 看到第 t+1…T-1 个未来 token，保证自回归生成的因果性
> - `key_padding_mask`：屏蔽目标序列中的 padding 位置，防止模型关注无效 token

In [ ]:
# ── 构造解码器自注意力所需的 MHA 实例 ──────────────────────────────────────
N_dec_sa, T_dec_sa, E_dec_sa = 2, 10, 256  # N=batch 大小=2, T=目标序列长=10, E=嵌入维度=256

mha_dec_sa = nn.MultiheadAttention(  # 解码器自注意力 MHA，参数与编码器相同
    embed_dim=E_dec_sa, num_heads=8, batch_first=True, dropout=0.0,
)  # 返回值：nn.MultiheadAttention 实例
mha_dec_sa.eval()  # 切换评估模式，关闭 Dropout

# ── 构造目标序列嵌入 ───────────────────────────────────────────────────────
tar_sa = torch.rand(N_dec_sa, T_dec_sa, E_dec_sa)  # 形状 (2, 10, 256)，dtype=float32，模拟 decoder 输入

# ── 构造因果掩码（attn_mask）─────────────────────────────────────────────────────
# generate_square_subsequent_mask 生成上三角 -inf 浮点掩码，.bool() 转换：-inf→True（忽略）, 0→False（可见）
# 形状 (T, T) = (10, 10)，dtype=bool；上三角（未来位置）为 True，下三角（已生成位置）为 False
tgt_attn_mask = nn.Transformer.generate_square_subsequent_mask(T_dec_sa).bool()  # 形状 (10, 10)
print('因果掩码形状:', tgt_attn_mask.shape)          # -> torch.Size([10, 10])
print('因果掩码（前 4×4）:
', tgt_attn_mask[:4, :4])  # 左下角 False（可见），右上角 True（未来，忽略）

# ── 构造 padding 掩码（key_padding_mask）─────────────────────────────────────────────
# 形状 (N, T) = (2, 10)，dtype=bool；True=该 token 是 padding，不参与自注意力
tgt_key_padding_mask_sa = torch.zeros(N_dec_sa, T_dec_sa, dtype=torch.bool)  # 初始全 False
tgt_key_padding_mask_sa[0, 8:] = True  # 第 0 个样本：位置 8、9 是 padding（该句只有 8 个词）
print('padding 掩码形状:', tgt_key_padding_mask_sa.shape)  # -> torch.Size([2, 10])

with torch.no_grad():  # 展示演示，不计算梯度
    dec_attn_out, _ = mha_dec_sa(
        query=tar_sa,                                  # Query 输入，形状 (N, T, E) = (2, 10, 256)
        key=tar_sa,                                    # Key 输入，形状 (N, T, E) = (2, 10, 256)
        value=tar_sa,                                  # Value 输入，形状 (N, T, E) = (2, 10, 256)
        attn_mask=tgt_attn_mask,                       # 因果掩码，形状 (T, T) = (10, 10)；True=不可见（未来 token）
        key_padding_mask=tgt_key_padding_mask_sa,      # padding 掩码，形状 (N, T) = (2, 10)；True=忽略
        need_weights=False,                            # bool；False 时权重返回 None
    )
    # 返回值 1—dec_attn_out：形状 (N, T, embed_dim) = (2, 10, 256)，dtype=float32

print('解码器自注意力输入形状:', tar_sa.shape)         # -> torch.Size([2, 10, 256])
print('解码器自注意力输出形状:', dec_attn_out.shape)    # -> torch.Size([2, 10, 256])
print('输入形状 = 输出形状？', tar_sa.shape == dec_attn_out.shape)  # -> True

#### 3.2.2 交叉注意力（Cross-Attention）

> 交叉注意力中，Query 来自 Decoder，Key/Value 来自 Encoder 输出。当两者维度不同时，需要配置 `kdim` 和 `vdim` 启用独立投影层。

In [ ]:
# ── 构造支持交叉注意力的 MHA（kdim/vdim 与 embed_dim 不同）─────────────
# Decoder Query 维度为 256，Encoder 输出维度为 512，两者不同
cross_mha = nn.MultiheadAttention(
    embed_dim=256,    # Query 所在空间维度（输出维度），int
    num_heads=8,      # 注意力头数，int；自注意力和交叉注意力均使用此头数
    kdim=512,         # Key 的输入特征维度，int；启用独立的 k_proj_weight (256, 512)
    vdim=512,         # Value 的输入特征维度，int；启用独立的 v_proj_weight (256, 512)
    batch_first=True, # bool；输入形状为 (N, L, E)
    dropout=0.0,      # float；评估演示无需 Dropout
)  # 返回值：nn.MultiheadAttention 实例
# 注：kdim != embed_dim 时会启用独立的 q_proj_weight/k_proj_weight/v_proj_weight 参数

cross_mha.eval()  # 切换评估模式

# ── 构造交叉注意力输入 ────────────────────────────────────────────────
N_ca, T_ca, S_ca = 2, 10, 15  # N=batch, T=Decoder Query 序列长, S=Encoder KV 序列长
E_q_ca, E_kv_ca = 256, 512    # Query 维度=256, Key/Value 维度=512

query_dec = torch.rand(N_ca, T_ca, E_q_ca)   # Decoder 目标序列嵌入，形状 (2, 10, 256)，dtype=float32
key_enc   = torch.rand(N_ca, S_ca, E_kv_ca)  # Encoder 输出 Key，形状 (2, 15, 512)，dtype=float32
value_enc = torch.rand(N_ca, S_ca, E_kv_ca)  # Encoder 输出 Value，形状 (2, 15, 512)，dtype=float32

with torch.no_grad():
    cross_out, _ = cross_mha(
        query=query_dec,   # Query 来自 Decoder，形状 (N, T, E_q) = (2, 10, 256)
        key=key_enc,       # Key 来自 Encoder，形状 (N, S, E_k) = (2, 15, 512)；通过 k_proj_weight 投影至 embed_dim
        value=value_enc,   # Value 来自 Encoder，形状 (N, S, E_v) = (2, 15, 512)；通过 v_proj_weight 投影至 embed_dim
        need_weights=False,
    )
    # 返回值：cross_out 形状 (N, T, embed_dim) = (2, 10, 256)，dtype=float32
    # 输出长度由 Query 序列长度 T=10 决定，每个位置融合了来自 Encoder 的上下文信息

print('交叉注意力— query 形状:', query_dec.shape)  # -> torch.Size([2, 10, 256])
print('交叉注意力— key 形状  :', key_enc.shape)    # -> torch.Size([2, 15, 512])
print('交叉注意力— value 形状:', value_enc.shape)  # -> torch.Size([2, 15, 512])
print('交叉注意力输出形状      :', cross_out.shape)   # -> torch.Size([2, 10, 256])

### 3.3 掩码（Mask）使用详解

> `nn.MultiheadAttention` 支持两类掩码同时使用：
> - **`key_padding_mask`**：描述「哪些 Key 位置是 padding」，与批数据相关（不同样本的 padding 位置不同）
> - **`attn_mask`**：描述「序列内部哪些位置对可见」，对 batch 内所有样本相同（如因果掩码）

> **掩码的 bool 语义**：`nn.MultiheadAttention` 中 bool 掩码 **True = 忽略/遮盖该位置**，这与 `F.scaled_dot_product_attention` 的 bool 语义完全相反（SDPA 中 True = 保留）。详见第五节。

In [ ]:
# ── 掩码演示环境准备 ──────────────────────────────────────────────────────
N_m, L_m, S_m, E_m = 2, 8, 10, 256  # batch=2, Query长度=8, Key长度=10, 嵌入维度=256

mha_m = nn.MultiheadAttention(  # 构造交叉注意力 MHA（L != S）
    embed_dim=E_m, num_heads=8, batch_first=True, dropout=0.0,
)
mha_m.eval()

query_m = torch.rand(N_m, L_m, E_m)  # Query 序列，形状 (2, 8, 256)，dtype=float32
key_m   = torch.rand(N_m, S_m, E_m)  # Key 序列，形状 (2, 10, 256)，dtype=float32
value_m = torch.rand(N_m, S_m, E_m)  # Value 序列，形状 (2, 10, 256)，dtype=float32

# ── 掩码一：key_padding_mask —— 说明头哪些 Key 位置是 padding ──────────────
# 形状 (N, S) = (2, 10)，dtype=bool
# True  = 该 Key 位置是 padding，注意力计算时将其分数设为 -inf（softmax 后趋近 0）
# False = 正常 token，参与注意力计算
key_padding_mask = torch.zeros(N_m, S_m, dtype=torch.bool)  # 初始全 False，所有位置均为正常 token
key_padding_mask[0, 8:] = True  # 第 0 个样本：Key 位置 8、9 是 padding（该句只有 8 个有效 token）

# ── 掩码二：attn_mask（float 类型）—— 直接加到注意力分数 ─────────────────
# 形状 (L, S) = (8, 10)，dtype=float32
# 常见用途：将解码器自注意力的因果掩码传入（L=S 时适用；此处 L != S，仅做形状演示）
attn_mask_float = torch.zeros(L_m, S_m)        # 全 0（所有位置可见），形状 (8, 10)，dtype=float32
attn_mask_float[:, 6:] = float('-inf')          # Key 位置 6、7、8、9 对所有 Query 不可见

# ── 掩码三：attn_mask（bool 类型）—— True 表示忽略小注意：与 float -inf 等价 ────────
# bool True = 遮盖（与 F.sdpa bool True=保留语义相反！）
attn_mask_bool = torch.zeros(L_m, S_m, dtype=torch.bool)  # 初始全 False（所有可见）
attn_mask_bool[:, 6:] = True                              # Key 位置 6+ 设为 True，即遮盖

print('key_padding_mask 形状:', key_padding_mask.shape)  # -> torch.Size([2, 10])
print('attn_mask_float 形状:', attn_mask_float.shape)    # -> torch.Size([8, 10])
print('attn_mask_bool  形状:', attn_mask_bool.shape)     # -> torch.Size([8, 10])

with torch.no_grad():
    out_masked, _ = mha_m(
        query=query_m,
        key=key_m,
        value=value_m,
        key_padding_mask=key_padding_mask,  # padding 掩码，形状 (N, S) = (2, 10)；True=忽略 Key
        attn_mask=attn_mask_float,          # 序列可见掩码，形状 (L, S) = (8, 10)；-inf=不可见
    )
    # 最终注意力分数 = QK^T/sqrt(d_k) + attn_mask_float（广播），并将 key_padding_mask 中 True 位置设为 -inf
print('带掩码的 MHA 输出形状:', out_masked.shape)  # -> torch.Size([2, 8, 256])

### 3.4 返回注意力权重

> 通过 `need_weights=True` 可获得注意力权重矩阵，用于可视化模型关注的位置。配合 `average_attn_weights` 可控制返回所有头的均值，还是每头独立权重。
>
> **注意**：`need_weights=True` 会额外多一次注意力权重的计算和内存拷贝，生产环境下若不需要权重应设为 `False`。

In [ ]:
# ── 注意力权重返回演示 ─────────────────────────────────────────────────────
N_w, L_w, E_w = 2, 6, 256   # batch=2, 序列长度=6, 嵌入维度=256
H_w = 4                      # 头数=4，每头维度 = 256//4 = 64

mha_w = nn.MultiheadAttention(
    embed_dim=E_w, num_heads=H_w, batch_first=True, dropout=0.0,
)
mha_w.eval()

x_w = torch.rand(N_w, L_w, E_w)  # 形状 (2, 6, 256)，dtype=float32，模拟自注意力输入

with torch.no_grad():
    # ── 方式一：average_attn_weights=True（默认）—— 返回各头均值权重 ───────────
    _, weights_avg = mha_w(
        x_w, x_w, x_w,           # Q=K=V=x_w，自注意力
        need_weights=True,        # bool；True = 计算并返回注意力权重
        average_attn_weights=True,# bool；True = 对所有 num_heads 头的注意力权重取均值
    )
    # weights_avg 形状：(N, L, S) = (2, 6, 6)，dtype=float32；[i, j, k] 表示样本 i 各头均值下第 j 个 Query 对第 k 个 Key 的注意力
    print('各头均值注意力权重形状:', weights_avg.shape)  # -> torch.Size([2, 6, 6])
    print('第 0 个样本平均权重矩阵（行=Query 位置, 列=Key 位置）:')
    print(weights_avg[0].round(decimals=3))   # 每行和为 1.0（softmax 归一化）

    # ── 方式二：average_attn_weights=False —— 返回每头独立注意力权重 ─────
    _, weights_per = mha_w(
        x_w, x_w, x_w,
        need_weights=True,
        average_attn_weights=False,  # bool；False = 返回每个头独立的注意力权重
    )
    # weights_per 形状：(N, H, L, S) = (2, 4, 6, 6)，dtype=float32；[i, h, j, k] 表示样本 i 第 h 头下第 j Query 对第 k Key 的注意力
    print()
    print('各头独立注意力权重形状:', weights_per.shape)  # -> torch.Size([2, 4, 6, 6])
    print(f'共 {weights_per.shape[1]} 个头，每头独立展示不同的关注模式')

## 四、F.scaled_dot_product_attention 详解

> `F.scaled_dot_product_attention`（SDPA）是 PyTorch 2.0 引入的底层注意力函数。
> - **输入已完成分头**：形状为 `(N, H, L, E_head)`，对应 `batch × heads × seq_len × head_dim`
> - **无可学习参数**：只负责 $\text{softmax}(QK^T / \sqrt{E_{head}} + \text{attn\_mask}) \times V$ 的计算
> - **自动选择内核**：Flash Attention、Memory-Efficient Attention 或标准内核，根据硬件和参数自动切换

### 4.1 接口说明与形状约定

> `F.scaled_dot_product_attention(query, key, value, attn_mask, dropout_p, is_causal, scale)` 的参数说明如下。

> ---

| 参数名 | 形状 | 类型 | 默认值 | 含义 |
|:---|:---:|:---:|:---:|:---|
| `query` | (N, H, L, E_h) | Tensor | — | 已分头的 Query 张量 |
| `key` | (N, H, S, E_h) | Tensor | — | 已分头的 Key 张量 |
| `value` | (N, H, S, E_v) | Tensor | — | 已分头的 Value 张量；E_v 可与 E_h 不同 |
| `attn_mask` | 可广播到 (N,H,L,S) | bool/float | None | **bool: True=保留, False=遮盖**（与 nn.MHA 相反！）；float: 直接加到注意力分数 |
| `dropout_p` | — | float | 0.0 | 注意力权重的 Dropout 概率；仅训练时设为 > 0.0 |
| `is_causal` | — | bool | False | True=自动生成上三角 -inf 因果掩码；与 attn_mask 互斥 |
| `scale` | — | float/None | None | 缩放因子；None = $1/\sqrt{E_h}$（默认） |

> ---

> **返回值**：Output 张量，形状 `(N, H, L, E_v)`，语义与输入 query 形状对应（序列长度由 L 决定）。

In [ ]:
# ── F.scaled_dot_product_attention 基础用法 ──────────────────────────────────
# SDPA 的输入是已完成分头的张量，形状为 (N, H, L, E_head)
# 实际使用时，Q/K/V 由线性投影 + reshape 得到

N_s, H_s, L_s, S_s, E_h = 2, 8, 10, 12, 32  # batch=2, heads=8, Q长=10, KV长=12, 每头维度=32

# 模拟已分头的 Q/K/V 张量（实际使用时由线性投影+reshape 得到）
query_s = torch.rand(N_s, H_s, L_s, E_h)  # 形状 (2, 8, 10, 32)，dtype=float32
key_s   = torch.rand(N_s, H_s, S_s, E_h)  # 形状 (2, 8, 12, 32)，dtype=float32
value_s = torch.rand(N_s, H_s, S_s, E_h)  # 形状 (2, 8, 12, 32)，dtype=float32

# ── 执行 SDPA：计算 softmax(QK^T / sqrt(E_head)) * V ───────────────────────
out_s = F.scaled_dot_product_attention(
    query_s,        # Query，形状 (N, H, L, E_h) = (2, 8, 10, 32)
    key_s,          # Key，形状 (N, H, S, E_h) = (2, 8, 12, 32)
    value_s,        # Value，形状 (N, H, S, E_v) = (2, 8, 12, 32)；E_v 可与 E_h 不同
    attn_mask=None, # None 表示无掩码，所有 Key 位置均可见
    dropout_p=0.0,  # float；注意力权重的 Dropout 概率；评估/演示时保持 0.0
    is_causal=False,# bool；False 表示不使用因果掩码（Encoder 层等全局注意力场景）
    scale=None,     # float 或 None；None = 默认 1/sqrt(E_h) = 1/sqrt(32) ≈ 0.177
)
# 返回值：out_s 形状 (N, H, L, E_v) = (2, 8, 10, 32)，dtype=float32
# 输出序列长度 L=10 由 Query 决定，每个位置融合了所有 Key/Value 的加权信息
print('SDPA 基础输出形状:', out_s.shape)  # -> torch.Size([2, 8, 10, 32])

# ── 验证自定义 scale 等价于默认 scale ──────────────────────────────
scale_val = 1.0 / math.sqrt(E_h)  # 手动计算默认 scale 值 = 1/sqrt(32)，返回 float
out_s2 = F.scaled_dot_product_attention(
    query_s, key_s, value_s,
    scale=scale_val,  # 显式传入手计算的 scale
)
print('自定义 scale 输出与默认 scale 的最大差异:',
      (out_s - out_s2).abs().max().item())  # -> 浮点误差级别，趋近 0

### 4.2 is_causal 因果掩码

> `is_causal=True` 时，SDPA 内部自动生成上三角 $-\infty$ 掩码，等价于手动传入下三角 bool True 的 `attn_mask`，但可额外支持 Flash Attention 加速。  
> **注意**：`is_causal=True` 与 `attn_mask` 互斥，无法同时使用。

In [ ]:
# ── is_causal=True 因果掩码演示 ──────────────────────────────────────────────
N_c2, H_c2, T_c2, E_c2 = 2, 4, 8, 64  # batch=2, heads=4, 序列长度=8, 每头维度=64

# 自注意力场景：Q=K=V，L=S=T_c2
query_c = torch.rand(N_c2, H_c2, T_c2, E_c2)  # 形状 (2, 4, 8, 64)，dtype=float32
key_c   = torch.rand(N_c2, H_c2, T_c2, E_c2)  # 形状 (2, 4, 8, 64)，dtype=float32
value_c = torch.rand(N_c2, H_c2, T_c2, E_c2)  # 形状 (2, 4, 8, 64)，dtype=float32

# ── 方式一：is_causal=True —— 自动生成因果掩码（推荐，支持 Flash Attention 加速）──────
out_causal = F.scaled_dot_product_attention(
    query_c, key_c, value_c,
    is_causal=True,  # bool；True = 自动生成上三角 -inf 掩码；位置 t 只能看到位置 0..t
)  # 返回值：形状 (N, H, T, E_h) = (2, 4, 8, 64)，dtype=float32

# ── 方式二：手动构造等价 bool 掩码（验证等价性） ─────────────────────────
# 注意！SDPA bool 掩码：True=保留, False=遮盖（与 nn.MHA 语义相反）
# 因果掩码：位置 (i,j) 处，j<=i 时可见（True），j>i 时不可见（False）
causal_mask = torch.ones(T_c2, T_c2, dtype=torch.bool).tril()  # 下三角为 True（保留），形状 (8, 8)，dtype=bool
out_manual = F.scaled_dot_product_attention(
    query_c, key_c, value_c,
    attn_mask=causal_mask,  # bool 掩码：True=保留，False=遮盖；形状 (8,8) 可广播到 (N,H,L,S)
    is_causal=False,        # 已手动提供掩码，不再使用自动因果掩码
)

# ── 验证两种方式结果一致 ──────────────────────────────────────────────────
diff = (out_causal - out_manual).abs().max().item()  # 最大绝对差，应为浮点误差级别
print('is_causal=True 与手动 bool 掩码结果最大差异:', diff)  # -> 接近 0
print('因果注意力输出形状:', out_causal.shape)             # -> torch.Size([2, 4, 8, 64])

# ── 展示手动 bool 掩码的形状 ──────────────────────────────────────────────
print('\n手动因果掩码（True=保留, False=遮盖）形状:', causal_mask.shape)  # -> torch.Size([8, 8])
print('前 4 行 4 列内容:')
print(causal_mask[:4, :4])  # 左下角 True（可见），右上角 False（不可见）

### 4.3 自定义 attn_mask（bool 与 float 两种形式）

> **float 掩码**：将 `attn_mask` 直接加到注意力分数上，`-inf` 表示不可见，`0.0` 表示无影响。  
> **bool 掩码**：内部自动转换为 float：`True → 0.0`（保留），`False → -inf`（遮盖）。

> **核心区别**：
> - `F.scaled_dot_product_attention` bool True = **保留**
> - `nn.MultiheadAttention` bool True = **忽略/遮盖**  
>
>  两者语义完全相反！混用会导致掩码效果完全反转。

In [ ]:
# ── SDPA attn_mask 两种形式演示：遮盖 Key 位置 6、7 ─────────────────────
# attn_mask 形状：(L, S) 或 (N, H, L, S) 或可广播到 (N, H, L, S) 的任意形状

N_a, H_a, L_a, S_a, E_a = 2, 4, 6, 8, 32  # batch=2, heads=4, Q长=6, K长=8, 每头维度=32

q_a = torch.rand(N_a, H_a, L_a, E_a)  # Query，形状 (2, 4, 6, 32)，dtype=float32
k_a = torch.rand(N_a, H_a, S_a, E_a)  # Key，  形状 (2, 4, 8, 32)，dtype=float32
v_a = torch.rand(N_a, H_a, S_a, E_a)  # Value，形状 (2, 4, 8, 32)，dtype=float32

# ── float 掩码：-inf 表示不可见 ────────────────────────────────────────────
float_mask = torch.zeros(L_a, S_a)         # 初始全 0.0（所有 Key 位置可见），形状 (6, 8)
float_mask[:, 6:] = float('-inf')           # Key 位置 6、7 不可见：直接加 -inf 加到分数

# ── bool 掩码：True=保留, False=遮盖（注意与 nn.MHA 语义相反！）───────────
bool_mask = torch.ones(L_a, S_a, dtype=torch.bool)  # 初始全 True（保留所有 Key），形状 (6, 8)
bool_mask[:, 6:] = False                             # Key 位置 6、7 设为 False（遮盖）
# 内部转换：False → -inf，True → 0.0，再加到注意力分数

out_f = F.scaled_dot_product_attention(q_a, k_a, v_a, attn_mask=float_mask)  # float 掩码
out_b = F.scaled_dot_product_attention(q_a, k_a, v_a, attn_mask=bool_mask)   # bool 掩码

print('float 掩码输出形状:', out_f.shape)              # -> torch.Size([2, 4, 6, 32])
print('bool  掩码输出形状:', out_b.shape)              # -> torch.Size([2, 4, 6, 32])
print('float/bool 掩码结果最大差异:',
      (out_f - out_b).abs().max().item())              # -> 浮点误差级别，证明两种形式等价

# ── 展示注意力语义对比 ────────────────────────────────────────────────────
print('\n== bool 掩码语义对比 ==')
print('[SDPA  ] bool True  = 保留（Key 可见），False = 遮盖（Key 不可见，-inf）')
print('[nn.MHA] bool True  = 忽略（Key 不可见），False = 保留（Key 可见）  <- 相反！')

## 五、两种接口的掩码 bool 语义差异

> 这是使用 PyTorch 注意力接口时最容易踩的陷阱，也是最常见的 Bug 来源：
> `nn.MultiheadAttention` 内部最终调用 `F.scaled_dot_product_attention`，但在调用之前会对 bool 掩码做一次取反（`~mask`），因此对外暴露的语义与 SDPA 内部语义相反。用户无需关心内部实现，只需记住：

> ---

| 接口 | bool True 的语义 |
|:---|:---|
| `nn.MultiheadAttention` (`attn_mask` / `key_padding_mask`) | True = **忽略/遮盖**该位置 |
| `F.scaled_dot_product_attention` (`attn_mask`) | True = **保留**该位置 |

> ---

> 下方代码通过构造一个可观察的小例子，验证两者掩码语义确实相反。

In [ ]:
# ── 掩码 bool 语义差异验证 ───────────────────────────────────────────────────
# 目标：构造一个简单例子，展示“仅保留 Key 位置 0”，观察两种接口下掩码写法相反

N_d, L_d, S_d, E_d = 1, 3, 4, 8  # batch=1, Q长=3, K长=4, 嵌入维度=8
H_d = 1                            # 头数=1，便于直观观察

# 构造简单的 Q/K/V——所有为 1 的张量，便于验证掩码效果
q_d = torch.ones(N_d, L_d, E_d)            # MHA 输入 Query，形状 (1, 3, 8)
k_d = torch.eye(S_d, E_d)[:S_d, :E_d]     # 形状 (4, 8)，每个 Key 位置不同
k_d = k_d.unsqueeze(0)                     # 形状 (1, 4, 8)，MHA 输入 Key
v_d = torch.arange(S_d, dtype=torch.float).reshape(1, S_d, 1).expand(1, S_d, E_d)  # Value: 每位置值不同

# 构造 MHA（投影权重为单位阵，消除投影层影响）
mha_d = nn.MultiheadAttention(
    embed_dim=E_d, num_heads=H_d, bias=False, batch_first=True, dropout=0.0,
)
mha_d.eval()
with torch.no_grad():
    nn.init.eye_(mha_d.in_proj_weight[:E_d])       # W_Q = 单位阵，投影不改变 Query
    nn.init.eye_(mha_d.in_proj_weight[E_d:2*E_d])  # W_K = 单位阵
    nn.init.eye_(mha_d.in_proj_weight[2*E_d:])     # W_V = 单位阵
    nn.init.eye_(mha_d.out_proj.weight)             # W_O = 单位阵

# ── 目标：仅保留 Key 位置 0，遮盖 Key 位置 1、2、3 ──────────────────────
# nn.MHA bool True = 忽略：保留位置 0 用 False，遮盖位置 1、2、3 用 True
kpm_mha  = torch.tensor([[False, True, True, True]])  # 形状 (1, 4)，dtype=bool
# F.sdpa  bool True = 保留：保留位置 0 用 True，遮盖位置 1、2、3 用 False
mask_sdpa = torch.tensor([[True, False, False, False]])  # 形状 (1, 4)，dtype=bool

print('相同效果下两种接口的 bool 掩码写法相反：')
print('nn.MHA  key_padding_mask :', kpm_mha)    # -> [[False, True, True, True]]
print('F.sdpa  attn_mask (bool) :', mask_sdpa)  # -> [[True, False, False, False]]
print('两者互为取反：', (~kpm_mha == mask_sdpa).all().item())  # -> True

# ── 分别用两种接口验证输出相同 ─────────────────────────────────────────
with torch.no_grad():
    out_mha, _ = mha_d(q_d, k_d, v_d, key_padding_mask=kpm_mha, need_weights=False)  # MHA
q_sdpa = q_d.view(N_d, H_d, L_d, E_d)       # 将 MHA 形状 (1,3,8) 转为 SDPA 形状 (1,1,3,8)；q_d 连续，view 安全
k_sdpa = k_d.view(N_d, H_d, S_d, E_d)       # 形状 (1, 1, 4, 8)；k_d 连续，view 安全
v_sdpa = v_d.reshape(N_d, H_d, S_d, E_d)    # 形状 (1, 1, 4, 8)；v_d 由 expand() 得到，stride=0 非连续
                                              # 必须用 reshape()，view() 对非连续张量会抛 RuntimeError
out_sdpa = F.scaled_dot_product_attention(q_sdpa, k_sdpa, v_sdpa, attn_mask=mask_sdpa)  # SDPA
out_sdpa = out_sdpa.view(N_d, L_d, E_d)  # 将输出形状转回 (1, 3, 8)

print('\nMHA 输出（取第 0 个 Query 位置，前 4 维）:', out_mha[0, 0, :4].tolist())
print('SDPA输出（取第 0 个 Query 位置，前 4 维）:', out_sdpa[0, 0, :4].tolist())
print('两种接口结果最大差异:', (out_mha - out_sdpa).abs().max().item())  # -> 超小浮点误差

## 六、接口对比总结

> **形状字母含义**：N = batch size；H = 头数（num_heads）；L = Query 序列长度；S = Key/Value 序列长度；E = 嵌入维度（embed_dim）；E_h = 每头维度（E // H）

> ---

| 属性 | nn.MultiheadAttention | F.scaled_dot_product_attention |
|:---|:---|:---|
| **模块类型** | `nn.Module`，含可学习参数 | 函数式 API，无可学习参数 |
| **Q/K/V 线性投影** | ✅ 内置（$W_Q, W_K, W_V, W_O$） | ❌ 不含，需用户自行投影后传入 |
| **Q 输入形状** (batch_first=True) | (N, L, E) | (N, H, L, E_h) 已分头 |
| **K/V 输入形状** (batch_first=True) | (N, S, E_k/E_v) | (N, H, S, E_h) 已分头 |
| **输出形状** | (N, L, E) | (N, H, L, E_h) |
| **bool 掩码 True 的语义** | **True = 忽略/遮盖** ❗ | **True = 保留** ❗（与左相反） |
| **padding 掩码** | `key_padding_mask (N, S)` 单独层 | 统一写入 `attn_mask` |
| **因果掩码** | `attn_mask (L, S)` 或 `is_causal=True` | `attn_mask (L, S)` 或 `is_causal=True` |
| **返回注意力权重** | ✅ `need_weights=True` 返回 (N,L,S) | ❌ 不返回 |
| **Flash Attention 支持** | ⚠️ `need_weights=True` 时禁用 | ✅ `is_causal=True` 时最优 |
| **典型使用场景** | 快速搭建标准 Transformer | 自定义注意力变体（RoPE、GQA、MQA 等）|

> ---

## 七、关键结论

> 1. **接口层次关系**：`nn.MultiheadAttention` 内部最终调用 `F.scaled_dot_product_attention`。前者是含线性投影的完整模块，后者是纯注意力计算的函数，两者职责划分清晰。
> 2. **接口的 bool 掩码语义必须分清楚**：
>    - `nn.MultiheadAttention`：bool **True = 忽略/遮盖**该位置（从 -inf 加到分数）
>    - `F.scaled_dot_product_attention`：bool **True = 保留**该位置（不影响分数）
>    - 两者语义**完全相反**！混用导致掩码效果全面反转，是最常见的 Bug 来源。
> 3. **输入形状不同**：`nn.MultiheadAttention` 输入未分头的 `(N, L, E)`，`F.sdpa` 输入已分头的 `(N, H, L, E_h)`。前者内部自动处理分头。
> 4. **决策设计建议**：
>    - 快速搭建标准模型：优先选择 `nn.MultiheadAttention`，注意力权重可视化段也可用
>    - 自定义注意力变体（RoPE、GQA、MQA、ALiBi 等）：用 `F.sdpa` 配合自定义投影层灵活实现
>    - 性能敏感场景：使用 `F.sdpa` + `is_causal=True`，充分利用 Flash Attention 内核加速